In [5]:
"""
Splits labeled_dataset_new.xlsx into:
  1. svm_train_80.xlsx        - 80% pool, full labels. Used to (re)train SVM,
                                 and as the source pool to hand-pick few-shot
                                 demonstration examples from.
  2. eval_holdout_20_unlabeled.csv - 20% held-out set, NO labels. Feed this
                                 into SVM, zero-shot, and few-shot inference.
  3. eval_holdout_20_ground_truth.csv - 20% held-out set, id + Label only.
                                 Keep this untouched by inference scripts;
                                 only used at scoring time.

Dedup rule: 4 ids appear twice with identical Label/label_reason but one
copy missing similarity_score (two merged export batches). We keep the
copy that HAS similarity_score when available.

Stratified 80/20 split on Label, random_state=42.
"""
import pandas as pd

SRC = "labeled_dataset_new_deduped.xlsx"
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/classification_redo"

df = pd.read_excel(SRC)
print(f"Loaded {len(df)} rows")

# --- Dedup: prefer the row with a non-null similarity_score ---
df["_has_sim"] = df["similarity_score"].notna()
df = (
    df.sort_values("_has_sim", ascending=False)
      .drop_duplicates(subset="id", keep="first")
      .drop(columns="_has_sim")
      .reset_index(drop=True)
)
print(f"After dedup: {len(df)} rows")
print(df["Label"].value_counts())

# --- Stratified 80/20 split ---
train_idx = []
for label, group in df.groupby("Label"):
    train_idx.extend(group.sample(frac=0.8, random_state=42).index.tolist())

train80 = df.loc[train_idx].reset_index(drop=True)
holdout20 = df.drop(index=train_idx).reset_index(drop=True)

print(f"\ntrain80: {len(train80)} rows")
print(train80["Label"].value_counts())
print(f"\nholdout20: {len(holdout20)} rows")
print(holdout20["Label"].value_counts())

assert set(train80["id"]).isdisjoint(set(holdout20["id"])), "Leakage: overlapping ids!"
assert len(train80) + len(holdout20) == len(df)

# --- Save outputs ---
train80.to_csv(f"{OUT_DIR}/svm_train_80.csv", index=False)

holdout20[["id", "text"]].to_csv(f"{OUT_DIR}/eval_holdout_20_unlabeled.csv", index=False)
holdout20[["id", "Label"]].to_csv(f"{OUT_DIR}/eval_holdout_20_ground_truth.csv", index=False)

print("\nSaved:")
print(f"  {OUT_DIR}/svm_train_80.csv")
print(f"  {OUT_DIR}/eval_holdout_20_unlabeled.csv")
print(f"  {OUT_DIR}/eval_holdout_20_ground_truth.csv")

Loaded 406 rows
After dedup: 406 rows
Label
0    208
1    198
Name: count, dtype: int64

train80: 324 rows
Label
0    166
1    158
Name: count, dtype: int64

holdout20: 82 rows
Label
0    42
1    40
Name: count, dtype: int64

Saved:
  /Users/nadia/Desktop/redditRun_june/classification_redo/svm_train_80.csv
  /Users/nadia/Desktop/redditRun_june/classification_redo/eval_holdout_20_unlabeled.csv
  /Users/nadia/Desktop/redditRun_june/classification_redo/eval_holdout_20_ground_truth.csv
